[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/11_random_forest/03_oob_importance_and_tuning.ipynb)

# Bài 03 — OOB score, feature importance, tuning và failure case

**🎯 Mục tiêu**
- Tính **out-of-bag (OOB) score** from scratch — "cross-validation miễn phí".
- Feature importance của Random Forest: MDI vs permutation; vấn đề **feature tương quan** chia nhau importance.
- Tuning: hyperparameter nào quan trọng; `n_estimators` không gây overfit; `RandomizedSearchCV`.
- Failure case: ngoại suy, dữ liệu thưa nhiều chiều, độ trễ & bộ nhớ, xác suất chưa calibrate; template model.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/11_random_forest

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RandomizedSearchCV, cross_val_score, train_test_split

from ml_course.datasets import make_customer_churn
from ml_course.visualization import set_style

set_style()
churn = make_customer_churn(n=5000, seed=11)
X = pd.get_dummies(churn.drop(columns=["customer_id", "churn"]), dtype=float)
y = churn["churn"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

## 1. OOB score from scratch

Mỗi cây không "thấy" ~36.8% mẫu (out-of-bag). Với mỗi mẫu $i$, lấy trung bình dự đoán **chỉ của những cây không train trên $i$** → một dự đoán "như trên dữ liệu mới" cho **mọi** mẫu train, không cần tách tập validation.

In [ ]:
def oob_predict_proba(X, y, n_estimators=200, max_features="sqrt", min_samples_leaf=3, seed=0):
    from sklearn.tree import DecisionTreeClassifier

    rng = np.random.default_rng(seed)
    n = len(X)
    votes, counts = np.zeros((n, 2)), np.zeros(n)
    for _ in range(n_estimators):
        idx = rng.integers(0, n, n)
        oob = np.setdiff1d(np.arange(n), idx)                     # mẫu cây này KHÔNG thấy
        tree = DecisionTreeClassifier(max_features=max_features, min_samples_leaf=min_samples_leaf,
                                      random_state=int(rng.integers(1 << 31))).fit(X[idx], y[idx])
        votes[oob] += tree.predict_proba(X[oob])
        counts[oob] += 1
    return votes[:, 1] / np.maximum(counts, 1), counts


p_oob, counts = oob_predict_proba(X_train.to_numpy(), y_train)
print(f"mỗi mẫu là OOB với trung bình {counts.mean():.1f}/200 cây (lý thuyết ≈ {200 * np.exp(-1):.1f})")
print(f"OOB accuracy (scratch) = {np.mean((p_oob >= 0.5) == y_train):.4f} | OOB ROC-AUC = {roc_auc_score(y_train, p_oob):.4f}")

rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, oob_score=True, random_state=0).fit(X_train, y_train)
cv_acc = cross_val_score(RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=0),
                         X_train, y_train, cv=5).mean()
print(f"sklearn oob_score_ = {rf.oob_score_:.4f} | 5-fold CV accuracy = {cv_acc:.4f} | test accuracy = {rf.score(X_test, y_test):.4f}")

OOB ≈ CV ≈ test, nhưng OOB chỉ cần **một lần** train. Hữu ích để chọn hyperparameter nhanh khi dữ liệu lớn. (Lưu ý: OOB hơi bi quan khi số cây ít — mỗi mẫu chỉ được ~37% số cây đánh giá.)

## 2. Feature importance và feature tương quan

Thêm một bản sao gần như giống hệt của `tenure_months` (`tenure_copy` = tenure + nhiễu nhỏ). Hai feature giờ **chia nhau** importance:

In [ ]:
rng = np.random.default_rng(0)
X2_train = X_train.assign(tenure_copy=X_train["tenure_months"] + rng.normal(0, 0.5, len(X_train)))
X2_test = X_test.assign(tenure_copy=X_test["tenure_months"] + rng.normal(0, 0.5, len(X_test)))
rf1 = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0).fit(X_train, y_train)
rf2 = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0).fit(X2_train, y_train)

rows = {}
for name, model, Xt in [("gốc", rf1, X_test), ("thêm tenure_copy", rf2, X2_test)]:
    perm = permutation_importance(model, Xt, y_test, n_repeats=5, random_state=0, scoring="roc_auc")
    rows[name] = pd.DataFrame({"MDI": model.feature_importances_, "permutation": perm.importances_mean}, index=Xt.columns)
pd.concat(rows, axis=1).loc[["tenure_months", "tenure_copy", "contract_month-to-month", "support_calls", "monthly_charges"]].round(4)

- **MDI**: importance của tenure **phân tán** sang hai cột — mỗi cột riêng lẻ trông kém quan trọng hơn tenure gốc (tổng hai cột còn bị thổi phồng: nhóm "tenure" giờ xuất hiện trong tập feature ngẫu nhiên của mỗi split thường gấp đôi).
- **Permutation**: còn tệ hơn — xáo trộn một cột thì model vẫn dùng được cột kia → cả hai đều trông **ít quan trọng** (mỗi cột chỉ còn ~1/3 importance của tenure gốc)!
- Khắc phục: gom nhóm feature tương quan rồi permute **cả nhóm**, loại bớt feature trùng lặp, hoặc dùng SHAP với hiểu biết về tương quan (chương 36). Đừng kết luận "feature không quan trọng" chỉ vì importance thấp.

## 3. Hyperparameter và tuning

| Hyperparameter | Vai trò | Gợi ý |
|---|---|---|
| `n_estimators` | số cây | càng nhiều càng ổn định (không overfit); 200–1000 |
| `max_features` | tập feature mỗi split | **quan trọng nhất**; `sqrt` (phân loại), 0.3–1.0 (hồi quy) |
| `min_samples_leaf` | kích thước lá | 1–20; lớn hơn → mượt hơn, xác suất tốt hơn |
| `max_depth` | độ sâu | thường để None, điều khiển bằng `min_samples_leaf` |
| `max_samples` | tỉ lệ mẫu bootstrap | < 1 → nhanh hơn, đa dạng hơn |
| `class_weight` | cân bằng lớp | `"balanced"`, `"balanced_subsample"` |

In [ ]:
ns = [1, 5, 10, 25, 50, 100, 200]
test = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, min_samples_leaf=3, random_state=0).fit(X_train, y_train)
    test.append(roc_auc_score(y_test, m.predict_proba(X_test)[:, 1]))
plt.semilogx(ns, test, "o-")
plt.xlabel("n_estimators")
plt.ylabel("ROC-AUC test")
plt.title("Thêm cây: tăng rồi bão hoà — không giảm")
plt.show()

In [ ]:
search = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=100, random_state=0),
    {"max_features": [1, 2, 3, "sqrt", 0.5, 0.8], "min_samples_leaf": [1, 3, 5, 10, 20, 40], "max_samples": [0.5, 0.8, None]},
    n_iter=12, cv=5, scoring="roc_auc", random_state=0).fit(X_train, y_train)
print("tốt nhất:", search.best_params_, f"| ROC-AUC CV = {search.best_score_:.4f}")
print(f"ROC-AUC test = {roc_auc_score(y_test, search.predict_proba(X_test)[:, 1]):.4f}")
pd.DataFrame(search.cv_results_)[["param_max_features", "param_min_samples_leaf", "param_max_samples", "mean_test_score"]] \
    .sort_values("mean_test_score", ascending=False).head(5).round(4)

**Extra Trees** (Extremely Randomized Trees): ngưỡng split cũng chọn **ngẫu nhiên** (không tìm ngưỡng tốt nhất) → nhanh hơn, đa dạng hơn, đôi khi tốt bằng hoặc hơn RF:

In [ ]:
for name, model in [("Random Forest", RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0)),
                    ("Extra Trees", ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, random_state=0))]:
    start = time.perf_counter()
    model.fit(X_train, y_train)
    print(f"{name:14s} ROC-AUC test = {roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]):.4f} | train {time.perf_counter() - start:.2f}s")

## 4. Failure case

**(a) Ngoại suy** — giống cây đơn lẻ: dự đoán là trung bình các giá trị đã thấy.

In [ ]:
t = np.arange(200).reshape(-1, 1)
sales = 100 + 1.5 * t.ravel() + np.random.default_rng(1).normal(0, 10, 200)
rf_ts = RandomForestRegressor(n_estimators=200, random_state=0).fit(t[:150], sales[:150])
print(f"RF dự đoán t = 199: {rf_ts.predict([[199]])[0]:.1f} | thật ≈ {100 + 1.5 * 199:.1f} | max y train = {sales[:150].max():.1f}")

**(b) Quan hệ tuyến tính cộng dồn từ nhiều feature** (và tương tự: dữ liệu thưa nhiều chiều như văn bản): mỗi split chỉ nhìn một feature, cây phải xấp xỉ một siêu phẳng 50 chiều bằng các hộp chữ nhật → cần rất nhiều dữ liệu. Model tuyến tính thắng rõ:

In [ ]:
from sklearn.linear_model import LogisticRegression

n, d = 1000, 50
X_lin = rng.normal(size=(n, d))
w = rng.normal(0, 0.5, d)
y_lin = (rng.random(n) < 1 / (1 + np.exp(-(X_lin @ w)))).astype(int)      # log-odds tuyến tính theo 50 feature
for name, model in [("Random Forest", RandomForestClassifier(n_estimators=300, random_state=0)),
                    ("Logistic Regression", LogisticRegression(max_iter=2000))]:
    print(f"{name:20s} ROC-AUC (5-fold CV) = {cross_val_score(model, X_lin, y_lin, cv=5, scoring='roc_auc').mean():.4f}")

**(c) Chi phí triển khai:** 500 cây sâu → hàng chục đến hàng trăm MB (tuỳ kích thước dữ liệu), độ trễ dự đoán lớn hơn nhiều so với model tuyến tính.

In [ ]:
import pickle

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

big = RandomForestClassifier(n_estimators=500, random_state=0).fit(X_train, y_train)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_train, y_train)
for name, model in [("RF 500 cây", big), ("Logistic Regression", lr)]:
    size = len(pickle.dumps(model)) / 1e3
    start = time.perf_counter()
    for _ in range(50):
        model.predict_proba(X_test.iloc[:1])
    latency = (time.perf_counter() - start) / 50 * 1000
    print(f"{name:20s} kích thước = {size:10,.1f} KB | độ trễ 1 dự đoán ≈ {latency:6.2f} ms")

**(d) Xác suất chưa calibrate:** trung bình phiếu của các cây thường bị "kéo vào giữa" (hiếm khi gần 0 hoặc 1) → cần calibration nếu dùng xác suất (chương 37).

## 5. Template model: Random Forest

| Phần | Random Forest |
|---|---|
| Problem | phân loại, hồi quy (cũng: phát hiện bất thường — Isolation Forest, ước lượng độ tương tự) |
| Intuition | nhiều cây đa dạng bỏ phiếu → giảm variance |
| Geometry | ranh giới bậc thang mượt hơn cây đơn (trung bình nhiều hộp) |
| Probability | tỉ lệ phiếu / trung bình xác suất lá — thường chưa calibrated |
| Objective | mỗi cây giảm impurity tham lam; rừng = trung bình (không tối ưu toàn cục) |
| Optimization | bootstrap + random feature subset mỗi split; song song hoá hoàn toàn |
| Assumptions | rất ít; cần các cây đủ tốt và ít tương quan |
| Complexity | train $O(B \cdot d_{try} \cdot n\log^2 n)$, song song; dự đoán $O(B \cdot \text{độ sâu})$ |
| Hyperparameters | `max_features` (quan trọng nhất), `min_samples_leaf`, `n_estimators`, `max_samples`, `class_weight` |
| Preprocessing | không cần scaling; encode categorical; xử lý missing (sklearn ≥ 1.4 hỗ trợ) |
| Strengths | mạnh "out of the box", ít cần tuning, OOB miễn phí, ít overfit, song song, robust với outlier/nhiễu |
| Weaknesses | không ngoại suy, model lớn & chậm khi dự đoán, kém với dữ liệu thưa nhiều chiều, importance bị lệch khi feature tương quan, kém gradient boosting khi tuning kỹ |
| Interpretation | permutation importance, partial dependence, SHAP (TreeSHAP) |

## 🧠 Kiểm tra nhanh

1. OOB score khác cross-validation thế nào? Ưu điểm?
2. Hai feature gần như trùng nhau: importance của chúng thay đổi thế nào? Kết luận "feature không quan trọng" có an toàn không?
3. Có nên tune `n_estimators` bằng CV không?
4. Vì sao Random Forest yếu với dữ liệu văn bản bag-of-words?

## 📌 Tóm tắt

- OOB = đánh giá miễn phí bằng các cây không thấy mẫu.
- Importance bị chia/che khi feature tương quan — diễn giải thận trọng.
- `max_features`, `min_samples_leaf` quan trọng nhất; nhiều cây không overfit.
- Failure: ngoại suy, thưa nhiều chiều, chi phí triển khai, xác suất chưa calibrate.

## 📚 Tài liệu tham khảo

- Breiman (2001) — *Random Forests*
- Geurts, Ernst & Wehenkel (2006) — *Extremely randomized trees*
- Strobl et al. (2008) — *Conditional variable importance for random forests*
- Probst, Wright & Boulesteix (2019) — *Hyperparameters and tuning strategies for random forest*